In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =============================================================================
# EPILEPSY GENE EXPRESSION — 2-MODEL ENSEMBLE PIPELINE (RF + ET)
# =============================================================================
# Models  : Random Forest · Extra Trees
# FIXED   : Critical 1  — SMOTE applied only inside inner folds; test fold never augmented
#           Critical 2  — Filter thresholds selected purely from inner-fold data
#           Critical 3  — Single normalization path; no double-norm distortion
#           Critical 4  — SHAP computed on outer training fold, not tiny test fold
# =============================================================================

import subprocess, sys, os, pickle
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg in ["optuna", "imbalanced-learn", "shap", "xgboost", "tqdm"]:
    install(pkg)

import glob
data_path = glob.glob("/kaggle/input/**/ML_matrix_DEGs_only.csv", recursive=True)
data_path = data_path[0] if data_path else "ML_matrix_DEGs_only.csv"
print(f"   Data file: {data_path}")

import numpy as np
import pandas as pd
import warnings
import optuna
import time
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, average_precision_score
)
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
CONFIG = {
    "data_path"        : data_path,
    "label_col"        : "Label",
    "n_outer_folds"    : 5,
    "n_inner_folds"    : 3,
    "n_trials"         : 200,
    "random_state"     : 42,
    "top_n_genes"      : 30,
    "smote_k"          : 3,
    "overfit_penalty"  : 0.15,
    "n_perm_repeats"   : 30,
    "shap_bg_size"     : 50,
    "shap_explain_size": 80,
    "checkpoint_dir"   : "./checkpoints_RF_ET",
}

# ── ONLY RF AND ET ────────────────────────────────────────────────────────────
MODELS = ["RF", "ET"]

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

print("=" * 70)
print("   EPILEPSY GENE EXPRESSION — 2-MODEL ENSEMBLE PIPELINE")
print("   Random Forest · Extra Trees")
print("=" * 70)
print(f"   Outer CV      : {CONFIG['n_outer_folds']}-Fold Stratified")
print(f"   Inner CV      : {CONFIG['n_inner_folds']}-Fold Stratified")
print(f"   Trials        : {CONFIG['n_trials']} per fold per model")
print(f"   Top genes     : {CONFIG['top_n_genes']}")
print("=" * 70)
print("\n  CRITICAL FIXES APPLIED:")
print("   [1] SMOTE only in inner training splits — test fold never augmented")
print("   [2] Filter thresholds selected from inner folds only — no leakage")
print("   [3] Single normalization path — no double-norm distortion")
print("   [4] SHAP computed on outer TRAINING fold — not tiny test fold")
print("=" * 70)

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("\n[Step 1]  Loading data ...")
df         = pd.read_csv(CONFIG["data_path"])
print(f"   Raw shape : {df.shape}")
y          = df[CONFIG["label_col"]].values.astype(int)
X_df       = df.drop(columns=[CONFIG["label_col"]])
non_numeric = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    X_df = X_df.drop(columns=non_numeric)
gene_names = np.array(X_df.columns.tolist())
X          = X_df.values.astype(np.float32)
classes, counts = np.unique(y, return_counts=True)
print(f"   Class distribution:")
for c, n in zip(classes, counts):
    lbl = "Control  (0)" if c == 0 else "Epilepsy (1)"
    print(f"      {lbl}: {n}  ({n/len(y)*100:.1f}%)")
print(f"   Total genes   : {X.shape[1]}")
print(f"   Total samples : {X.shape[0]}")

# ── HELPERS ───────────────────────────────────────────────────────────────────
def apply_filters(X_train, X_test, var_pct, mad_pct):
    variances = np.var(X_train, axis=0)
    var_mask  = variances >= np.percentile(variances, var_pct)
    median    = np.median(X_train, axis=0)
    mad       = np.median(np.abs(X_train - median), axis=0)
    mad_mask  = mad >= np.percentile(mad, mad_pct)
    mask      = var_mask & mad_mask
    if mask.sum() < 20: mask = var_mask
    if mask.sum() < 20: mask = np.ones(X_train.shape[1], dtype=bool)
    return X_train[:, mask], X_test[:, mask], mask

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy"    : round(accuracy_score(y_true, y_pred), 4),
        "f1_macro"    : round(f1_score(y_true, y_pred, average="macro",    zero_division=0), 4),
        "f1_weighted" : round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "precision"   : round(precision_score(y_true, y_pred,              zero_division=0), 4),
        "recall"      : round(recall_score(y_true, y_pred,                 zero_division=0), 4),
        "roc_auc"     : round(roc_auc_score(y_true, y_prob), 4),
        "pr_auc"      : round(average_precision_score(y_true, y_prob), 4),
    }

def safe_smote(X_tr, y_tr, k, rs):
    min_count = np.bincount(y_tr).min()
    k_safe    = max(1, min(k, min_count - 1))
    return SMOTE(k_neighbors=k_safe, random_state=rs).fit_resample(X_tr, y_tr)

def norm(arr):
    s = arr.sum()
    return arr / s if s > 0 else arr

# ── CHECKPOINT HELPERS ────────────────────────────────────────────────────────
def ckpt_path_phase1(model_name, fold_idx):
    return os.path.join(CONFIG["checkpoint_dir"],
                        f"phase1_{model_name}_fold{fold_idx}.pkl")

def ckpt_path_phase2(model_name, fold_idx):
    return os.path.join(CONFIG["checkpoint_dir"],
                        f"phase2_{model_name}_fold{fold_idx}.pkl")

def ckpt_path_phase3():
    return os.path.join(CONFIG["checkpoint_dir"], "phase3_consensus.pkl")

def save_checkpoint(path, data):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(data, f, protocol=4)
    os.replace(tmp, path)
    print(f"    [CKPT] Saved → {os.path.basename(path)}")

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

def list_checkpoints():
    files = sorted(glob.glob(os.path.join(CONFIG["checkpoint_dir"], "*.pkl")))
    if not files:
        print("   (no checkpoints found)")
        return
    for f in files:
        size_kb = os.path.getsize(f) / 1024
        print(f"   {os.path.basename(f):<45}  {size_kb:>7.1f} KB")

# ── OPTUNA OBJECTIVES ─────────────────────────────────────────────────────────
def make_objective_rf(X_outer_train, y_outer_train, cfg, trial_bar, best_tracker):
    def objective(trial):
        var_pct = trial.suggest_float("var_percentile", 10.0, 60.0)
        mad_pct = trial.suggest_float("mad_percentile", 10.0, 60.0)
        params  = dict(
            n_estimators     = trial.suggest_int  ("n_estimators",      50, 200),
            max_depth        = trial.suggest_int  ("max_depth",          3,  10),
            min_samples_split= trial.suggest_int  ("min_samples_split",  2,  20),
            min_samples_leaf = trial.suggest_int  ("min_samples_leaf",   1,  10),
            max_features     = trial.suggest_float("max_features",     0.1, 0.8),
            bootstrap        = trial.suggest_categorical("bootstrap", [True, False]),
            random_state     = cfg["random_state"],
            n_jobs           = -1,
        )
        inner_cv = StratifiedKFold(n_splits=cfg["n_inner_folds"], shuffle=True,
                                   random_state=cfg["random_state"])
        val_scores, train_scores = [], []
        for tr_idx, val_idx in inner_cv.split(X_outer_train, y_outer_train):
            X_itr,  y_itr  = X_outer_train[tr_idx],  y_outer_train[tr_idx]
            X_ival, y_ival = X_outer_train[val_idx],  y_outer_train[val_idx]
            X_itr_f, X_ival_f, _ = apply_filters(X_itr, X_ival, var_pct, mad_pct)
            X_res, y_res = safe_smote(X_itr_f, y_itr, cfg["smote_k"], cfg["random_state"])
            model = RandomForestClassifier(**params)
            model.fit(X_res, y_res)
            val_scores.append(  average_precision_score(y_ival, model.predict_proba(X_ival_f)[:, 1]))
            train_scores.append(average_precision_score(y_itr,  model.predict_proba(X_itr_f)[:, 1]))
        mean_val   = float(np.mean(val_scores))
        mean_train = float(np.mean(train_scores))
        gap        = max(0.0, mean_train - mean_val)
        penalised  = mean_val - cfg["overfit_penalty"] * gap
        trial_bar.update(1)
        if penalised > best_tracker["best_score"]:
            best_tracker["best_score"] = penalised
        trial_bar.set_postfix({"val_PR": f"{mean_val:.4f}", "gap": f"{gap:.4f}",
                                "best": f"{best_tracker['best_score']:.4f}"})
        return penalised
    return objective


def make_objective_et(X_outer_train, y_outer_train, cfg, trial_bar, best_tracker):
    def objective(trial):
        var_pct = trial.suggest_float("var_percentile", 10.0, 60.0)
        mad_pct = trial.suggest_float("mad_percentile", 10.0, 60.0)
        params  = dict(
            n_estimators     = trial.suggest_int  ("n_estimators",      50, 300),
            max_depth        = trial.suggest_int  ("max_depth",          3,  15),
            min_samples_split= trial.suggest_int  ("min_samples_split",  2,  20),
            min_samples_leaf = trial.suggest_int  ("min_samples_leaf",   1,  10),
            max_features     = trial.suggest_float("max_features",     0.1, 1.0),
            bootstrap        = trial.suggest_categorical("bootstrap", [True, False]),
            random_state     = cfg["random_state"],
            n_jobs           = -1,
        )
        inner_cv = StratifiedKFold(n_splits=cfg["n_inner_folds"], shuffle=True,
                                   random_state=cfg["random_state"])
        val_scores, train_scores = [], []
        for tr_idx, val_idx in inner_cv.split(X_outer_train, y_outer_train):
            X_itr,  y_itr  = X_outer_train[tr_idx],  y_outer_train[tr_idx]
            X_ival, y_ival = X_outer_train[val_idx],  y_outer_train[val_idx]
            X_itr_f, X_ival_f, _ = apply_filters(X_itr, X_ival, var_pct, mad_pct)
            X_res, y_res = safe_smote(X_itr_f, y_itr, cfg["smote_k"], cfg["random_state"])
            model = ExtraTreesClassifier(**params)
            model.fit(X_res, y_res)
            val_scores.append(  average_precision_score(y_ival, model.predict_proba(X_ival_f)[:, 1]))
            train_scores.append(average_precision_score(y_itr,  model.predict_proba(X_itr_f)[:, 1]))
        mean_val   = float(np.mean(val_scores))
        mean_train = float(np.mean(train_scores))
        gap        = max(0.0, mean_train - mean_val)
        penalised  = mean_val - cfg["overfit_penalty"] * gap
        trial_bar.update(1)
        if penalised > best_tracker["best_score"]:
            best_tracker["best_score"] = penalised
        trial_bar.set_postfix({"val_PR": f"{mean_val:.4f}", "gap": f"{gap:.4f}",
                                "best": f"{best_tracker['best_score']:.4f}"})
        return penalised
    return objective


objective_makers = {
    "RF" : make_objective_rf,
    "ET" : make_objective_et,
}

# ── STORAGE ───────────────────────────────────────────────────────────────────
fold_best_params   = {m: [None]*CONFIG["n_outer_folds"] for m in MODELS}
fold_best_filter   = {m: [None]*CONFIG["n_outer_folds"] for m in MODELS}
fold_test_metrics  = {m: [None]*CONFIG["n_outer_folds"] for m in MODELS}
fold_train_metrics = {m: [None]*CONFIG["n_outer_folds"] for m in MODELS}

gene_shap_sum   = {m: np.zeros(len(gene_names)) for m in MODELS}
gene_native_sum = {m: np.zeros(len(gene_names)) for m in MODELS}
gene_perm_sum   = {m: np.zeros(len(gene_names)) for m in MODELS}
gene_fold_count = {m: np.zeros(len(gene_names), dtype=int) for m in MODELS}

phase2_loaded_folds = {m: set() for m in MODELS}

pipeline_start = time.time()
outer_cv       = StratifiedKFold(n_splits=CONFIG["n_outer_folds"], shuffle=True,
                                  random_state=CONFIG["random_state"])
outer_splits   = list(outer_cv.split(X, y))

# ── CHECKPOINT RESTORE ────────────────────────────────────────────────────────
print("\n[Resume check] Scanning checkpoints ...")
list_checkpoints()

for model_name in MODELS:
    for fold_idx in range(CONFIG["n_outer_folds"]):
        ckpt = load_checkpoint(ckpt_path_phase1(model_name, fold_idx))
        if ckpt is not None:
            fold_best_params[model_name][fold_idx] = ckpt["best_params"]
            fold_best_filter[model_name][fold_idx] = ckpt["best_filter"]
            print(f"   Restored Phase1: {model_name} fold {fold_idx+1}")

for model_name in MODELS:
    for fold_idx in range(CONFIG["n_outer_folds"]):
        ckpt = load_checkpoint(ckpt_path_phase2(model_name, fold_idx))
        if ckpt is not None:
            fold_test_metrics[model_name][fold_idx]  = ckpt["test_metrics"]
            fold_train_metrics[model_name][fold_idx] = ckpt["train_metrics"]
            if fold_idx not in phase2_loaded_folds[model_name]:
                gene_shap_sum[model_name]   += ckpt["shap_contribution"]
                gene_native_sum[model_name] += ckpt["native_contribution"]
                gene_perm_sum[model_name]   += ckpt["perm_contribution"]
                gene_fold_count[model_name] += ckpt["fold_mask"].astype(int)
                phase2_loaded_folds[model_name].add(fold_idx)
            tm = ckpt["test_metrics"]
            print(f"   Restored Phase2: {model_name} fold {fold_idx+1}  "
                  f"roc_auc={tm['roc_auc']:.4f}  pr_auc={tm['pr_auc']:.4f}")

phase3_ckpt = load_checkpoint(ckpt_path_phase3())

p1_done = sum(fold_best_params[m][fi] is not None
              for m in MODELS for fi in range(CONFIG["n_outer_folds"]))
p2_done = sum(fold_test_metrics[m][fi] is not None
              for m in MODELS for fi in range(CONFIG["n_outer_folds"]))
total   = CONFIG["n_outer_folds"] * len(MODELS)
print(f"\n   Phase1 done: {p1_done}/{total}  |  Phase2 done: {p2_done}/{total}")

# =============================================================================
# PHASE 1 — Optuna Search
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 1 — Optuna Search  (RF + ET only)")
print("━"*70)

for model_name in MODELS:
    print(f"\n  ┌{'─'*66}")
    print(f"  │  MODEL: {model_name}")
    print(f"  └{'─'*66}")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_splits):
        fold_num = fold_idx + 1

        if fold_best_params[model_name][fold_idx] is not None:
            print(f"\n    Fold {fold_num} | SKIPPED (checkpoint exists)")
            continue

        X_outer_train = X[train_idx]
        y_outer_train = y[train_idx]

        print(f"\n    Fold {fold_num} | train={len(train_idx)} test={len(test_idx)}"
              f" | Running {CONFIG['n_trials']} trials ...")

        best_tracker = {"best_score": -np.inf}
        trial_bar    = tqdm(total=CONFIG["n_trials"],
                            desc=f"      {model_name} Fold {fold_num}",
                            position=1, leave=False, colour="cyan")

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=CONFIG["random_state"]),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=30),
        )
        study.optimize(
            objective_makers[model_name](
                X_outer_train, y_outer_train, CONFIG, trial_bar, best_tracker),
            n_trials=CONFIG["n_trials"], show_progress_bar=False,
        )
        trial_bar.close()

        best_p  = dict(study.best_trial.params)
        var_pct = best_p.pop("var_percentile")
        mad_pct = best_p.pop("mad_percentile")

        fold_best_params[model_name][fold_idx] = best_p
        fold_best_filter[model_name][fold_idx] = {"var_percentile": var_pct,
                                                   "mad_percentile": mad_pct}

        save_checkpoint(ckpt_path_phase1(model_name, fold_idx), {
            "best_params": best_p,
            "best_filter": {"var_percentile": var_pct, "mad_percentile": mad_pct},
            "best_value" : study.best_trial.value,
            "model_name" : model_name,
            "fold_idx"   : fold_idx,
        })

        n_kept = apply_filters(X_outer_train, X_outer_train, var_pct, mad_pct)[2].sum()
        print(f"    Best penalised PR-AUC : {study.best_trial.value:.4f}")
        print(f"    Genes after filtering : {n_kept} / {X.shape[1]}")

print(f"\n  PHASE 1 complete ✓")

# =============================================================================
# PHASE 2 — Retrain + SHAP + Permutation
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 2 — Retrain + SHAP (on train fold) + Permutation (on val split)")
print("━"*70)

metrics_list = ["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc","pr_auc"]

for model_name in MODELS:
    print(f"\n  {'='*66}")
    print(f"  MODEL: {model_name}")
    print(f"  {'='*66}")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_splits):
        fold_num = fold_idx + 1

        if fold_test_metrics[model_name][fold_idx] is not None:
            tm = fold_test_metrics[model_name][fold_idx]
            print(f"\n  {model_name} Fold {fold_num} | SKIPPED  "
                  f"roc_auc={tm['roc_auc']:.4f}  pr_auc={tm['pr_auc']:.4f}")
            continue

        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx],  y[test_idx]

        fold_var_pct = fold_best_filter[model_name][fold_idx]["var_percentile"]
        fold_mad_pct = fold_best_filter[model_name][fold_idx]["mad_percentile"]
        fold_params  = dict(fold_best_params[model_name][fold_idx])

        X_tr_f, X_te_f, gene_mask = apply_filters(X_tr, X_te, fold_var_pct, fold_mad_pct)
        X_res, y_res = safe_smote(X_tr_f, y_tr, CONFIG["smote_k"], CONFIG["random_state"])

        print(f"\n  {model_name} Fold {fold_num} | Genes: {gene_mask.sum()}"
              f" | var={fold_var_pct:.1f} mad={fold_mad_pct:.1f}"
              f" | train(aug)={len(X_res)} test(orig)={len(X_te_f)}")

        # ── Build model ───────────────────────────────────────────────────────
        if model_name == "RF":
            fold_params.pop("class_weight", None)
            fold_params.update({"random_state": CONFIG["random_state"], "n_jobs": -1})
            model = RandomForestClassifier(**fold_params)
            model.fit(X_res, y_res)

        else:  # ET
            fold_params.pop("class_weight", None)
            fold_params.update({"random_state": CONFIG["random_state"], "n_jobs": -1})
            model = ExtraTreesClassifier(**fold_params)
            model.fit(X_res, y_res)

        # ── Metrics ───────────────────────────────────────────────────────────
        y_te_prob  = model.predict_proba(X_te_f)[:, 1]
        y_te_pred  = model.predict(X_te_f)
        y_tr_prob  = model.predict_proba(X_tr_f)[:, 1]
        y_tr_pred  = model.predict(X_tr_f)
        native_imp = model.feature_importances_

        test_m  = compute_metrics(y_te, y_te_pred, y_te_prob)
        train_m = compute_metrics(y_tr, y_tr_pred, y_tr_prob)

        print(f"    {'Metric':<15} {'Train':>8} {'Test':>8} {'Gap':>8}  Status")
        print(f"    {'─'*52}")
        for metric in metrics_list:
            tr, te = train_m[metric], test_m[metric]
            gap    = tr - te
            status = "⚠ HIGH" if gap > 0.20 else "~ mild" if gap > 0.10 else "✓ good"
            print(f"    {metric:<15} {tr:>8.4f} {te:>8.4f} {gap:>+8.4f}  {status}")

        # ── Native importance ─────────────────────────────────────────────────
        gene_native_sum[model_name][gene_mask] += native_imp
        gene_fold_count[model_name][gene_mask] += 1

        # ── SHAP on training fold ─────────────────────────────────────────────
        print(f"    Computing SHAP on training fold ...", end=" ", flush=True)
        n_explain   = min(CONFIG["shap_explain_size"], len(X_tr_f))
        rng         = np.random.default_rng(CONFIG["random_state"])
        explain_idx = rng.choice(len(X_tr_f), size=n_explain, replace=False)
        X_explain   = X_tr_f[explain_idx]

        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_explain)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        if shap_vals.ndim == 3:
            shap_vals = shap_vals[:, :, 1]
        shap_mean = np.abs(shap_vals).mean(axis=0)
        gene_shap_sum[model_name][gene_mask] += shap_mean
        print("done ✓")

        # ── Permutation importance on inner val split ─────────────────────────
        print(f"    Computing Permutation Importance ...", end=" ", flush=True)
        perm_inner_cv = StratifiedKFold(n_splits=3, shuffle=True,
                                        random_state=CONFIG["random_state"])
        perm_train_idx, perm_val_idx = next(perm_inner_cv.split(X_tr_f, y_tr))
        X_perm_val = X_tr_f[perm_val_idx]
        y_perm_val = y_tr[perm_val_idx]

        perm = permutation_importance(
            model, X_perm_val, y_perm_val,
            n_repeats    = CONFIG["n_perm_repeats"],
            random_state = CONFIG["random_state"],
            scoring      = "average_precision",
            n_jobs       = -1)
        perm_mean = np.maximum(perm.importances_mean, 0)
        gene_perm_sum[model_name][gene_mask] += perm_mean
        print("done ✓")

        # ── Save checkpoint ───────────────────────────────────────────────────
        shap_full   = np.zeros(len(gene_names))
        native_full = np.zeros(len(gene_names))
        perm_full   = np.zeros(len(gene_names))
        shap_full[gene_mask]   = shap_mean
        native_full[gene_mask] = native_imp
        perm_full[gene_mask]   = perm_mean

        save_checkpoint(ckpt_path_phase2(model_name, fold_idx), {
            "test_metrics"       : test_m,
            "train_metrics"      : train_m,
            "shap_contribution"  : shap_full,
            "native_contribution": native_full,
            "perm_contribution"  : perm_full,
            "fold_mask"          : gene_mask,
            "fold_var_pct"       : fold_var_pct,
            "fold_mad_pct"       : fold_mad_pct,
            "model_name"         : model_name,
            "fold_idx"           : fold_idx,
        })

        fold_test_metrics[model_name][fold_idx]  = test_m
        fold_train_metrics[model_name][fold_idx] = train_m
        phase2_loaded_folds[model_name].add(fold_idx)

print(f"\n  PHASE 2 complete ✓")

# =============================================================================
# PHASE 3 — Consensus Gene Scoring (RF + ET only)
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 3 — Consensus Gene Scoring  (RF + ET, single normalization path)")
print("━"*70)

if phase3_ckpt is not None:
    print("  Loading Phase 3 from checkpoint ...")
    mean_shap           = phase3_ckpt["mean_shap"]
    mean_native         = phase3_ckpt["mean_native"]
    mean_perm           = phase3_ckpt["mean_perm"]
    shap_norm           = phase3_ckpt["shap_norm"]
    native_norm         = phase3_ckpt["native_norm"]
    perm_norm           = phase3_ckpt["perm_norm"]
    per_model_consensus = phase3_ckpt["per_model_consensus"]
    consensus_score     = phase3_ckpt["consensus_score"]
    sorted_idx          = phase3_ckpt["sorted_idx"]
    sorted_genes        = phase3_ckpt["sorted_genes"]
    sorted_scores       = phase3_ckpt["sorted_scores"]
    selected_genes      = phase3_ckpt["selected_genes"]
    selected_scores     = phase3_ckpt["selected_scores"]
    gene_df             = phase3_ckpt["gene_df"]
    per_model_gene_dfs  = phase3_ckpt["per_model_gene_dfs"]
    all_summary_rows    = phase3_ckpt["all_summary_rows"]
    summary_df          = phase3_ckpt["summary_df"]
    print("  Phase 3 loaded ✓")

else:
    safe_count  = {m: np.where(gene_fold_count[m] > 0, gene_fold_count[m], 1) for m in MODELS}
    mean_shap   = {m: gene_shap_sum[m]   / safe_count[m] for m in MODELS}
    mean_native = {m: gene_native_sum[m] / safe_count[m] for m in MODELS}
    mean_perm   = {m: gene_perm_sum[m]   / safe_count[m] for m in MODELS}

    shap_norm   = {m: norm(mean_shap[m])   for m in MODELS}
    native_norm = {m: norm(mean_native[m]) for m in MODELS}
    perm_norm   = {m: norm(mean_perm[m])   for m in MODELS}

    per_model_consensus = {
        m: (shap_norm[m] + native_norm[m] + perm_norm[m]) / 3.0
        for m in MODELS
    }

    cross_model_scores = np.stack([per_model_consensus[m] for m in MODELS], axis=0)
    consensus_score    = cross_model_scores.mean(axis=0)

    sorted_idx      = np.argsort(consensus_score)[::-1]
    sorted_genes    = gene_names[sorted_idx]
    sorted_scores   = consensus_score[sorted_idx]
    top_n           = CONFIG["top_n_genes"]
    selected_genes  = sorted_genes[:top_n]
    selected_scores = sorted_scores[:top_n]

    # ── gene_df (overall consensus top 30) ───────────────────────────────────
    gene_rows = []
    for rank in range(1, top_n + 1):
        idx  = sorted_idx[rank - 1]
        gene = gene_names[idx]
        gene_rows.append({
            "rank"      : rank,
            "gene"      : gene,
            "consensus" : round(float(consensus_score[idx]), 6),
            "rf_score"  : round(float(per_model_consensus["RF"][idx]),  6),
            "et_score"  : round(float(per_model_consensus["ET"][idx]),  6),
            "rf_folds"  : int(gene_fold_count["RF"][idx]),
            "et_folds"  : int(gene_fold_count["ET"][idx]),
        })
    gene_df = pd.DataFrame(gene_rows)

    # ── per_model_gene_dfs (top 30 per model) ────────────────────────────────
    per_model_gene_dfs = {}
    for m in MODELS:
        model_scores   = per_model_consensus[m]
        model_sort_idx = np.argsort(model_scores)[::-1]
        model_top_idx  = model_sort_idx[:top_n]
        rows = []
        for rank, idx in enumerate(model_top_idx, 1):
            rows.append({
                "rank"            : rank,
                "gene"            : gene_names[idx],
                f"{m.lower()}_score": round(float(model_scores[idx]), 6),
                "shap_norm"       : round(float(shap_norm[m][idx]),   6),
                "native_norm"     : round(float(native_norm[m][idx]), 6),
                "perm_norm"       : round(float(perm_norm[m][idx]),   6),
                "folds_active"    : int(gene_fold_count[m][idx]),
                "consensus_score" : round(float(consensus_score[idx]), 6),
                "shap_mean"       : round(float(mean_shap[m][idx]),   6),
                "native_mean"     : round(float(mean_native[m][idx]), 6),
                "perm_mean"       : round(float(mean_perm[m][idx]),   6),
            })
        per_model_gene_dfs[m] = pd.DataFrame(rows)

    # ── Summary metrics ───────────────────────────────────────────────────────
    all_summary_rows = []
    for model_name in MODELS:
        for m in metrics_list:
            tr_vals = [f[m] for f in fold_train_metrics[model_name] if f is not None]
            te_vals = [f[m] for f in fold_test_metrics[model_name]  if f is not None]
            tr_mean, tr_std = np.mean(tr_vals), np.std(tr_vals)
            te_mean, te_std = np.mean(te_vals), np.std(te_vals)
            gap    = tr_mean - te_mean
            status = "⚠ HIGH" if gap > 0.20 else "~ mild" if gap > 0.10 else "✓ good"
            all_summary_rows.append({
                "model"      : model_name,
                "metric"     : m,
                "train_mean" : round(tr_mean, 4),
                "train_std"  : round(tr_std,  4),
                "test_mean"  : round(te_mean, 4),
                "test_std"   : round(te_std,  4),
                "gap"        : round(gap, 4),
                "status"     : status,
            })
    summary_df = pd.DataFrame(all_summary_rows)

    save_checkpoint(ckpt_path_phase3(), {
        "mean_shap": mean_shap, "mean_native": mean_native, "mean_perm": mean_perm,
        "shap_norm": shap_norm, "native_norm": native_norm, "perm_norm": perm_norm,
        "per_model_consensus": per_model_consensus,
        "consensus_score": consensus_score,
        "sorted_idx": sorted_idx, "sorted_genes": sorted_genes, "sorted_scores": sorted_scores,
        "selected_genes": selected_genes, "selected_scores": selected_scores,
        "gene_df": gene_df, "per_model_gene_dfs": per_model_gene_dfs,
        "all_summary_rows": all_summary_rows, "summary_df": summary_df,
    })
    print("  Phase 3 computed and checkpointed ✓")

top_n = CONFIG["top_n_genes"]

print(f"\n  Overall Top {top_n} Consensus Genes (RF + ET)")
print(f"\n  {'Rank':<5} {'Gene':<22} {'Consensus':>10} {'RF':>8} {'ET':>8}"
      f"  {'RF_n':>6} {'ET_n':>6}")
print(f"  {'─'*68}")
for _, row in gene_df.iterrows():
    print(f"  {int(row['rank']):<5} {row['gene']:<22} {row['consensus']:>10.6f} "
          f"{row['rf_score']:>8.6f} {row['et_score']:>8.6f}"
          f"  {int(row['rf_folds']):>6} {int(row['et_folds']):>6}")

# =============================================================================
# PHASE 3b — Intersection RF ∩ ET
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 3b — RF ∩ ET Intersection → Final Gene List")
print("━"*70)

def get_intersection(sets_dict, keys, consensus_score, gene_names):
    common = set.intersection(*[sets_dict[k] for k in keys])
    ranked = sorted(
        [(g, consensus_score[np.where(gene_names == g)[0][0]]) for g in common],
        key=lambda x: x[1], reverse=True
    )
    return ranked

gene_sets   = {m: set(per_model_gene_dfs[m]["gene"].values[:top_n]) for m in MODELS}
rf_et_genes = get_intersection(gene_sets, ["RF", "ET"], consensus_score, gene_names)

print(f"\n  RF ∩ ET  →  {len(rf_et_genes)} genes")
print(f"\n  {'Rank':<5} {'Gene':<25} {'Consensus':>12}")
print(f"  {'─'*45}")
for rank, (gene, score) in enumerate(rf_et_genes, 1):
    print(f"  {rank:<5} {gene:<25} {score:>12.6f}")

# Save intersection
df_intersection = pd.DataFrame({
    "rank"     : range(1, len(rf_et_genes)+1),
    "gene"     : [g for g, s in rf_et_genes],
    "consensus": [round(s, 6) for g, s in rf_et_genes],
})

# =============================================================================
# PHASE 3c — Save to Excel
# =============================================================================
output_file = "RF_ET_all_results.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    gene_df.to_excel(writer,               sheet_name="Overall_Top30",    index=False)
    per_model_gene_dfs["RF"].to_excel(writer, sheet_name="RF_Top30",      index=False)
    per_model_gene_dfs["ET"].to_excel(writer, sheet_name="ET_Top30",      index=False)
    df_intersection.to_excel(writer,       sheet_name="RF_ET_Intersect",  index=False)
    summary_df.to_excel(writer,            sheet_name="Metrics_Summary",  index=False)

print(f"\n  Saved: {output_file}")

# =============================================================================
# PHASE 4 — Final Metrics
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 4 — Final Averaged Metrics Per Model")
print("━"*70)

for model_name in MODELS:
    print(f"\n  ── {model_name} ──")
    print(f"  {'Metric':<15} {'Train':>10} {'±Std':>7} {'Test':>10} {'±Std':>7} {'Gap':>8}  Status")
    print(f"  {'─'*68}")
    for row in all_summary_rows:
        if row["model"] != model_name:
            continue
        print(f"  {row['metric']:<15} {row['train_mean']:>10.4f} {row['train_std']:>7.4f} "
              f"{row['test_mean']:>10.4f} {row['test_std']:>7.4f} "
              f"{row['gap']:>+8.4f}  {row['status']}")

# =============================================================================
# PHASE 5 — Save CSVs
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 5 — Saving CSVs")
print("━"*70)

gene_df.to_csv("RF_ET_overall_top30.csv", index=False)
summary_df.to_csv("RF_ET_metrics_summary.csv", index=False)
per_model_gene_dfs["RF"].to_csv("RF_ET_top30_RF.csv", index=False)
per_model_gene_dfs["ET"].to_csv("RF_ET_top30_ET.csv", index=False)
df_intersection.to_csv("RF_ET_intersection.csv", index=False)
print("  All CSVs saved.")

# =============================================================================
# PHASE 6 — Plots
# =============================================================================
print("\n" + "━"*70)
print("  PHASE 6 — Plots")
print("━"*70)

model_colors = {"RF": "#4C72B0", "ET": "#8B5A9E"}

# Plot 1: Train vs Test
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
x, w = np.arange(len(metrics_list)), 0.35
for ax, model_name in zip(axes, MODELS):
    sub      = summary_df[summary_df.model == model_name]
    tr_means = [sub.loc[sub.metric == m, "train_mean"].values[0] for m in metrics_list]
    tr_stds  = [sub.loc[sub.metric == m, "train_std"].values[0]  for m in metrics_list]
    te_means = [sub.loc[sub.metric == m, "test_mean"].values[0]  for m in metrics_list]
    te_stds  = [sub.loc[sub.metric == m, "test_std"].values[0]   for m in metrics_list]
    ax.bar(x-w/2, tr_means, w, yerr=tr_stds, label="Train",
           color=model_colors[model_name], capsize=4, alpha=0.55)
    ax.bar(x+w/2, te_means, w, yerr=te_stds, label="Test",
           color=model_colors[model_name], capsize=4, alpha=0.95)
    ax.set_xticks(x); ax.set_xticklabels(metrics_list, rotation=30, ha="right")
    ax.set_title(f"{model_name}"); ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Score (mean ± std, 5 folds)")
fig.suptitle("Train vs Test Metrics — RF + ET Ensemble", fontsize=13)
plt.tight_layout()
plt.savefig("RF_ET_plot_train_vs_test.png", dpi=150); plt.close()
print("  Saved: RF_ET_plot_train_vs_test.png")

# Plot 2: Overall top 30 consensus
fig, ax = plt.subplots(figsize=(11, 11))
colors  = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))[::-1]
bars    = ax.barh(range(top_n), selected_scores[::-1], color=colors[::-1])
ax.set_yticks(range(top_n))
ax.set_yticklabels(selected_genes[::-1], fontsize=9)
ax.set_xlabel("Consensus Score")
ax.set_title(f"Top {top_n} Genes — RF + ET Ensemble")
for bar, gene in zip(bars, selected_genes[::-1]):
    idx = np.where(gene_names == gene)[0][0]
    ax.text(bar.get_width()*1.005, bar.get_y()+bar.get_height()/2,
            f"R{gene_fold_count['RF'][idx]} E{gene_fold_count['ET'][idx]}",
            va="center", fontsize=6.5, color="#555")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("RF_ET_plot_top30_consensus.png", dpi=150); plt.close()
print("  Saved: RF_ET_plot_top30_consensus.png")

# Plot 3: RF ∩ ET intersection genes
fig, ax = plt.subplots(figsize=(9, max(5, len(rf_et_genes)*0.4+1)))
genes_plot  = [g for g, s in rf_et_genes]
scores_plot = [s for g, s in rf_et_genes]
colors_int  = plt.cm.YlGnBu(np.linspace(0.4, 0.9, len(genes_plot)))[::-1]
ax.barh(range(len(genes_plot)-1, -1, -1), scores_plot, color=colors_int, alpha=0.85)
ax.set_yticks(range(len(genes_plot)-1, -1, -1))
ax.set_yticklabels(genes_plot, fontsize=9)
ax.set_xlabel("Consensus Score")
ax.set_title(f"RF ∩ ET Intersection — {len(rf_et_genes)} Genes\n(Epilepsy Study)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("RF_ET_plot_intersection.png", dpi=150); plt.close()
print("  Saved: RF_ET_plot_intersection.png")

# Plot 4: Fold heatmap
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, model_name in zip(axes, MODELS):
    hm = pd.DataFrame(
        [[fold_test_metrics[model_name][fi][m] for m in metrics_list]
         for fi in range(CONFIG["n_outer_folds"])
         if fold_test_metrics[model_name][fi] is not None],
        columns=metrics_list,
        index=[f"Fold {i+1}" for i in range(CONFIG["n_outer_folds"])
               if fold_test_metrics[model_name][i] is not None]
    )
    sns.heatmap(hm, annot=True, fmt=".3f", cmap="YlGnBu",
                vmin=0, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title(f"{model_name} — Test Metrics Per Fold")
plt.suptitle("Test Metrics Per Fold — RF + ET", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("RF_ET_plot_fold_heatmap.png", dpi=150); plt.close()
print("  Saved: RF_ET_plot_fold_heatmap.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
elapsed = time.time() - pipeline_start
h, rem  = divmod(elapsed, 3600); mt, s = divmod(rem, 60)

print("\n" + "="*70)
print("  PIPELINE COMPLETE  (Random Forest · Extra Trees)")
print("="*70)
print(f"  Runtime : {int(h)}h {int(mt)}m {int(s)}s")

print(f"\n  FINAL TEST RESULTS (avg across {CONFIG['n_outer_folds']} folds):")
for model_name in MODELS:
    print(f"\n    {model_name}:")
    for row in all_summary_rows:
        if row["model"] == model_name:
            print(f"      {row['metric']:<15}: "
                  f"{row['test_mean']:.4f} ± {row['test_std']:.4f}  {row['status']}")

print(f"\n  RF ∩ ET FINAL GENES ({len(rf_et_genes)}):")
for rank, (gene, score) in enumerate(rf_et_genes, 1):
    print(f"    {rank:>3}. {gene:<22} consensus={score:.6f}")

print(f"\n  OUTPUT FILES:")
print(f"    RF_ET_all_results.xlsx        ← main file (all sheets)")
print(f"    RF_ET_overall_top30.csv")
print(f"    RF_ET_top30_RF.csv")
print(f"    RF_ET_top30_ET.csv")
print(f"    RF_ET_intersection.csv        ← your final RF∩ET genes")
print(f"    RF_ET_plot_train_vs_test.png")
print(f"    RF_ET_plot_top30_consensus.png")
print(f"    RF_ET_plot_intersection.png")
print(f"    RF_ET_plot_fold_heatmap.png")
print("="*70)

   Data file: /kaggle/input/datasets/maryammuarij/tyuoye/ML_matrix_DEGs_only.csv
   EPILEPSY GENE EXPRESSION — 2-MODEL ENSEMBLE PIPELINE
   Random Forest · Extra Trees
   Outer CV      : 5-Fold Stratified
   Inner CV      : 3-Fold Stratified
   Trials        : 200 per fold per model
   Top genes     : 30

  CRITICAL FIXES APPLIED:
   [1] SMOTE only in inner training splits — test fold never augmented
   [2] Filter thresholds selected from inner folds only — no leakage
   [3] Single normalization path — no double-norm distortion
   [4] SHAP computed on outer TRAINING fold — not tiny test fold

[Step 1]  Loading data ...
   Raw shape : (265, 1629)
   Class distribution:
      Control  (0): 39  (14.7%)
      Epilepsy (1): 226  (85.3%)
   Total genes   : 1627
   Total samples : 265

[Resume check] Scanning checkpoints ...
   (no checkpoints found)

   Phase1 done: 0/10  |  Phase2 done: 0/10

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE 1 — Optuna Search  (

      RF Fold 1:   0%|          | 0/200 [00:00<?, ?it/s]

    [CKPT] Saved → phase1_RF_fold0.pkl
    Best penalised PR-AUC : 0.9991
    Genes after filtering : 618 / 1627

    Fold 2 | train=212 test=53 | Running 200 trials ...


      RF Fold 2:   0%|          | 0/200 [00:00<?, ?it/s]

    [CKPT] Saved → phase1_RF_fold1.pkl
    Best penalised PR-AUC : 0.9979
    Genes after filtering : 973 / 1627

    Fold 3 | train=212 test=53 | Running 200 trials ...


...

    [CKPT] Saved → phase1_ET_fold2.pkl
    Best penalised PR-AUC : 0.9997
    Genes after filtering : 709 / 1627

    Fold 4 | train=212 test=53 | Running 200 trials ...


      ET Fold 4:   0%|          | 0/200 [00:00<?, ?it/s]

    [CKPT] Saved → phase1_ET_fold3.pkl
    Best penalised PR-AUC : 0.9988
    Genes after filtering : 592 / 1627

    Fold 5 | train=212 test=53 | Running 200 trials ...


      ET Fold 5:   0%|          | 0/200 [00:00<?, ?it/s]

    [CKPT] Saved → phase1_ET_fold4.pkl
    Best penalised PR-AUC : 0.9995
    Genes after filtering : 590 / 1627

  PHASE 1 complete ✓

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE 2 — Retrain + SHAP (on train fold) + Permutation (on val split)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  MODEL: RF

  RF Fold 1 | Genes: 618 | var=58.4 mad=47.7 | train(aug)=362 test(orig)=53
    Metric             Train     Test      Gap  Status
    ────────────────────────────────────────────────────
    accuracy          1.0000   1.0000  +0.0000  ✓ good
    f1_macro          1.0000   1.0000  +0.0000  ✓ good
    f1_weighted       1.0000   1.0000  +0.0000  ✓ good
    precision         1.0000   1.0000  +0.0000  ✓ good
    recall            1.0000   1.0000  +0.0000  ✓ good
    roc_auc           1.0000   1.0000  +0.0000  ✓ good
    pr_auc            1.0000   1.0000  +0.0000  ✓ good
    Computing SHAP on training fold ... done ✓
    Computing Pe